# Consultas para stakeholders
Entender el comportamiento operativo y extraer conclusiones accionables.

## ¿Qué clientes están comprando títulos por encima del precio de mercado en más de un 2%? ¿Con qué frecuencia ocurre este comportamiento?

In [0]:
%sql
SELECT 
    dc.id_cliente,
    COUNT(*) AS total_ops,
    SUM(CASE WHEN fo.desvio_pct > 0.02 AND fo.tipo_op = 'Compra' THEN 1 ELSE 0 END) AS compras_caras,
    ROUND(
        SUM(CASE WHEN fo.desvio_pct > 0.02 AND fo.tipo_op = 'Compra' THEN 1 ELSE 0 END) 
        / COUNT(*), 4
    ) AS frecuencia
FROM gold_byma.fact_operaciones fo
JOIN gold_byma.dim_cliente dc 
    ON fo.cliente_sk = dc.cliente_sk
GROUP BY dc.id_cliente
HAVING compras_caras > 0
ORDER BY frecuencia DESC

## ¿Cuáles son los instrumentos con mayor desvío promedio entre el precio operado y el precio de mercado?

In [0]:
%sql
SELECT 
    di.simbolo,
    di.descripcion_titulo,
    ROUND(AVG(fo.desvio_pct), 4) AS avg_desvio
FROM gold_byma.fact_operaciones fo
JOIN gold_byma.dim_instrumento di 
    ON fo.instrumento_sk = di.instrumento_sk
WHERE fo.desvio_pct IS NOT NULL
GROUP BY di.simbolo, di.descripcion_titulo
ORDER BY avg_desvio DESC
LIMIT 10

## ¿Qué canal genera el mayor volumen operado? Analizar separadamente ARS y USD.

In [0]:
%sql
WITH operaciones_con_ars AS (
    SELECT 
        fo.id_transaccion,
        fo.canal_sk,
        fo.monto_total_usd,
        df.fecha,
        fd.dolar_ccl,
        fo.monto_total_usd * fd.dolar_ccl AS monto_total_ars
    FROM gold_byma.fact_operaciones fo

    JOIN gold_byma.dim_fecha df 
        ON fo.fecha_sk = df.fecha_sk

    LEFT JOIN silver_byma.fecha_dolar fd 
        ON df.fecha = fd.fecha
)

SELECT 
    dc.nombre_canal,
    
    SUM(o.monto_total_usd) AS volumen_usd,
    SUM(o.monto_total_ars) AS volumen_ars

FROM operaciones_con_ars o

JOIN gold_byma.dim_canal dc 
    ON o.canal_sk = dc.canal_sk

GROUP BY dc.nombre_canal
ORDER BY volumen_ars DESC

## ¿Cómo evoluciona semana a semana la proporción de compras vs ventas para los instrumentos AL30 y AL30D?

In [0]:
%sql
SELECT 
    df.anio,
    WEEKOFYEAR(df.fecha) AS semana,
    di.simbolo,
    SUM(CASE WHEN fo.tipo_op = 'Compra' THEN 1 ELSE 0 END) AS compras,
    SUM(CASE WHEN fo.tipo_op = 'Venta' THEN 1 ELSE 0 END) AS ventas,
    ROUND(
        SUM(CASE WHEN fo.tipo_op = 'Compra' THEN 1 ELSE 0 END) /
        NULLIF(COUNT(*),0), 4
    ) AS ratio_compra
FROM gold_byma.fact_operaciones fo
JOIN gold_byma.dim_fecha df 
    ON fo.fecha_sk = df.fecha_sk
JOIN gold_byma.dim_instrumento di 
    ON fo.instrumento_sk = di.instrumento_sk
WHERE di.simbolo IN ('AL30', 'AL30D')
GROUP BY df.anio, semana, di.simbolo
ORDER BY df.anio, semana

## ¿Quiénes son los clientes más activos y en qué instrumentos concentran su operatoria?

In [0]:
%sql
WITH top_clientes AS (
    SELECT 
        cliente_sk,
        COUNT(*) AS total_ops
    FROM gold_byma.fact_operaciones
    GROUP BY cliente_sk
    ORDER BY total_ops DESC
    LIMIT 10
)

SELECT 
    dc.id_cliente,
    di.simbolo,
    COUNT(*) AS ops
FROM gold_byma.fact_operaciones fo
JOIN top_clientes tc 
    ON fo.cliente_sk = tc.cliente_sk
JOIN gold_byma.dim_cliente dc 
    ON fo.cliente_sk = dc.cliente_sk
JOIN gold_byma.dim_instrumento di 
    ON fo.instrumento_sk = di.instrumento_sk
GROUP BY dc.id_cliente, di.simbolo
ORDER BY dc.id_cliente, ops DESC

## BONUS: ¿Qué segmento de actividad (LOW / MEDIUM / HIGH) tiene mayor proporción de operaciones desfavorables?

In [0]:
%sql
SELECT 
    dc.segmento_actividad,
    COUNT(*) AS total_ops,
    SUM(CASE WHEN fo.desvio_pct < 0 THEN 1 ELSE 0 END) AS ops_perdida,
    ROUND(
        SUM(CASE WHEN fo.desvio_pct < 0 THEN 1 ELSE 0 END) / COUNT(*), 4
    ) AS ratio_perdida
FROM gold_byma.fact_operaciones fo
JOIN gold_byma.dim_cliente dc 
    ON fo.cliente_sk = dc.cliente_sk
GROUP BY dc.segmento_actividad
ORDER BY ratio_perdida DESC

## BONUS: ¿Existen clientes con alta frecuencia de operaciones y alto desvío respecto al mercado?

In [0]:
%sql
SELECT 
    dc.id_cliente,
    COUNT(*) AS total_ops,
    ROUND(AVG(ABS(fo.desvio_pct)), 4) AS avg_desvio
FROM gold_byma.fact_operaciones fo
JOIN gold_byma.dim_cliente dc 
    ON fo.cliente_sk = dc.cliente_sk
GROUP BY dc.id_cliente
HAVING total_ops > 50
ORDER BY avg_desvio DESC